# 🎯 Complete Emotion Detection Training + Visualization\n\n## Features:\n- ✅ Train model from FER2013 dataset\n- ✅ Show training progress with graphs\n- ✅ Display accuracy/loss curves\n- ✅ Show confusion matrix\n- ✅ Per-emotion accuracy statistics\n- ✅ Test with images\n- ✅ Save trained model\n\n**Dataset:** FER2013 Enhanced (35,887 images, 7 emotions)\n**Expected Accuracy:** 60-70% (can reach 90%+ with advanced techniques)

In [ ]:
# ============================================================================\n# CELL 1: IMPORTS AND SETUP\n# ============================================================================\nprint('🚀 Importing libraries...\\n')\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport cv2\nimport os\nfrom datetime import datetime\n\nimport tensorflow as tf\nfrom tensorflow.keras.models import Sequential\nfrom tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten, BatchNormalization\nfrom tensorflow.keras.optimizers import Adam\nfrom tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau\nfrom tensorflow.keras.utils import to_categorical\nfrom tensorflow.keras.preprocessing.image import ImageDataGenerator\n\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import classification_report, confusion_matrix, accuracy_score\n\n# Display settings\n%matplotlib inline\nplt.style.use('default')\nsns.set_palette('husl')\nplt.rcParams['figure.figsize'] = (14, 6)\n\n# Global variables\nEMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']\nIMG_SIZE = 48\n\nprint('✅ All libraries imported!')\nprint(f'TensorFlow: {tf.__version__}')\nprint(f'NumPy: {np.__version__}')\nprint(f'\\n💡 Ready to train model!')

In [ ]:
# ============================================================================\n# CELL 2: LOAD FER2013 DATASET\n# ============================================================================\nprint('📊 Loading FER2013 Enhanced Dataset...\\n')\n\n# Try multiple paths\ndataset_paths = [\n    'emotion_datasets/fer2013/fer2013_enhanced.csv',\n    'fer2013_enhanced.csv',\n    '../emotion_datasets/fer2013/fer2013_enhanced.csv'\n]\n\ndf = None\nfor path in dataset_paths:\n    if os.path.exists(path):\n        df = pd.read_csv(path)\n        print(f'✅ Loaded {len(df):,} samples from {path}')\n        break\n\nif df is None:\n    raise FileNotFoundError('❌ Dataset not found! Please ensure fer2013_enhanced.csv exists')\n\n# Display dataset info\nprint(f'\\n📈 Dataset Statistics:')\nprint(f'Total samples: {len(df):,}')\nprint(f'Columns: {list(df.columns)}')\n\n# Emotion distribution\nprint(f'\\n📊 Emotion Distribution:')\nemotion_counts = df['emotion'].value_counts()\nfor emotion, count in emotion_counts.items():\n    percentage = (count / len(df)) * 100\n    print(f'  {emotion:10s}: {count:6,} ({percentage:5.2f}%)')\n\n# Visualize distribution\nplt.figure(figsize=(12, 5))\n\nplt.subplot(1, 2, 1)\nemotion_counts.plot(kind='bar', color='skyblue')\nplt.title('Emotion Distribution (Count)', fontsize=14, fontweight='bold')\nplt.xlabel('Emotion')\nplt.ylabel('Count')\nplt.xticks(rotation=45)\nplt.grid(axis='y', alpha=0.3)\n\nplt.subplot(1, 2, 2)\nemotion_counts.plot(kind='pie', autopct='%1.1f%%', startangle=90)\nplt.title('Emotion Distribution (Percentage)', fontsize=14, fontweight='bold')\nplt.ylabel('')\n\nplt.tight_layout()\nplt.show()\n\nprint('\\n✅ Dataset loaded successfully!')

In [ ]:
# ============================================================================\n# CELL 3: PREPROCESS DATA\n# ============================================================================\nprint('🔧 Preprocessing data...\\n')\n\npixels = []\nlabels = []\nskipped = 0\n\nfor idx, row in df.iterrows():\n    try:\n        # Parse pixel string\n        pixel_values = [int(p) for p in str(row['pixels']).split()]\n        \n        if len(pixel_values) != IMG_SIZE * IMG_SIZE:\n            skipped += 1\n            continue\n        \n        # Reshape to image\n        pixel_array = np.array(pixel_values, dtype='uint8').reshape(IMG_SIZE, IMG_SIZE)\n        \n        # Apply histogram equalization\n        pixel_array = cv2.equalizeHist(pixel_array)\n        \n        pixels.append(pixel_array)\n        labels.append(row['emotion'])\n    except:\n        skipped += 1\n\nprint(f'⚠️ Skipped {skipped} invalid samples')\n\n# Convert to numpy arrays\nX = np.array(pixels, dtype='float32')\nX = X / 255.0  # Normalize\nX = X.reshape(-1, IMG_SIZE, IMG_SIZE, 1)  # Add channel dimension\n\n# Encode labels\nemotion_mapping = {emotion: idx for idx, emotion in enumerate(EMOTIONS)}\ny = np.array([emotion_mapping.get(emotion, 0) for emotion in labels])\ny = to_categorical(y, len(EMOTIONS))\n\nprint(f'✅ Preprocessed data:')\nprint(f'   X shape: {X.shape}')\nprint(f'   y shape: {y.shape}')\nprint(f'   Valid samples: {len(X):,}')\n\n# Split data (70% train, 15% val, 15% test)\nprint(f'\\n✂️ Splitting data...')\nX_train, X_temp, y_train, y_temp = train_test_split(\n    X, y, test_size=0.3, random_state=42, stratify=np.argmax(y, axis=1)\n)\nX_val, X_test, y_val, y_test = train_test_split(\n    X_temp, y_temp, test_size=0.5, random_state=42, stratify=np.argmax(y_temp, axis=1)\n)\n\nprint(f'Training samples: {len(X_train):,}')\nprint(f'Validation samples: {len(X_val):,}')\nprint(f'Test samples: {len(X_test):,}')\n\n# Show sample images\nprint(f'\\n📸 Sample Images:')\nfig, axes = plt.subplots(2, 7, figsize=(14, 4))\nfor i, emotion in enumerate(EMOTIONS):\n    # Find first sample of this emotion\n    idx = np.where(np.argmax(y_train, axis=1) == i)[0][0]\n    axes[0, i].imshow(X_train[idx].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')\n    axes[0, i].set_title(emotion, fontsize=10, fontweight='bold')\n    axes[0, i].axis('off')\n    \n    # Show another sample\n    idx2 = np.where(np.argmax(y_train, axis=1) == i)[0][1]\n    axes[1, i].imshow(X_train[idx2].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')\n    axes[1, i].axis('off')\n\nplt.suptitle('Sample Images from Each Emotion', fontsize=14, fontweight='bold')\nplt.tight_layout()\nplt.show()\n\nprint('\\n✅ Data preprocessing complete!')

In [ ]:
# ============================================================================\n# CELL 4: CREATE CNN MODEL\n# ============================================================================\nprint('🏗️ Creating CNN Model...\\n')\n\nmodel = Sequential([\n    # Block 1\n    Conv2D(64, (3, 3), padding='same', input_shape=(IMG_SIZE, IMG_SIZE, 1)),\n    BatchNormalization(),\n    tf.keras.layers.Activation('relu'),\n    Conv2D(64, (3, 3), padding='same'),\n    BatchNormalization(),\n    tf.keras.layers.Activation('relu'),\n    MaxPooling2D(pool_size=(2, 2)),\n    Dropout(0.25),\n    \n    # Block 2\n    Conv2D(128, (3, 3), padding='same'),\n    BatchNormalization(),\n    tf.keras.layers.Activation('relu'),\n    Conv2D(128, (3, 3), padding='same'),\n    BatchNormalization(),\n    tf.keras.layers.Activation('relu'),\n    MaxPooling2D(pool_size=(2, 2)),\n    Dropout(0.25),\n    \n    # Block 3\n    Conv2D(256, (3, 3), padding='same'),\n    BatchNormalization(),\n    tf.keras.layers.Activation('relu'),\n    Conv2D(256, (3, 3), padding='same'),\n    BatchNormalization(),\n    tf.keras.layers.Activation('relu'),\n    MaxPooling2D(pool_size=(2, 2)),\n    Dropout(0.25),\n    \n    # Dense layers\n    Flatten(),\n    Dense(512),\n    BatchNormalization(),\n    tf.keras.layers.Activation('relu'),\n    Dropout(0.5),\n    \n    Dense(256),\n    BatchNormalization(),\n    tf.keras.layers.Activation('relu'),\n    Dropout(0.3),\n    \n    # Output layer\n    Dense(len(EMOTIONS), activation='softmax')\n])\n\n# Compile model\nmodel.compile(\n    optimizer=Adam(learning_rate=0.0001),\n    loss='categorical_crossentropy',\n    metrics=['accuracy']\n)\n\nprint('✅ Model created successfully!')\nprint(f'📊 Total parameters: {model.count_params():,}')\nprint(f'\\n🏗️ Model Architecture:')\nmodel.summary()

In [ ]:
# ============================================================================\n# CELL 5: SETUP TRAINING CONFIGURATION\n# ============================================================================\nprint('⚙️ Setting up training configuration...\\n')\n\n# Data augmentation\ndatagen = ImageDataGenerator(\n    rotation_range=20,\n    width_shift_range=0.15,\n    height_shift_range=0.15,\n    shear_range=0.15,\n    zoom_range=0.15,\n    horizontal_flip=True,\n    fill_mode='nearest'\n)\ndatagen.fit(X_train)\n\n# Callbacks\ntimestamp = datetime.now().strftime('%Y%m%d_%H%M%S')\nmodel_name = f'emotion_model_{timestamp}'\n\ncallbacks = [\n    ModelCheckpoint(\n        f'{model_name}_best.h5',\n        monitor='val_accuracy',\n        save_best_only=True,\n        mode='max',\n        verbose=1\n    ),\n    EarlyStopping(\n        monitor='val_accuracy',\n        patience=15,\n        restore_best_weights=True,\n        verbose=1\n    ),\n    ReduceLROnPlateau(\n        monitor='val_loss',\n        factor=0.5,\n        patience=5,\n        min_lr=1e-7,\n        verbose=1\n    )\n]\n\n# Training parameters\nEPOCHS = 50\nBATCH_SIZE = 32\n\nprint('✅ Configuration complete!')\nprint(f'\\n📋 Training Settings:')\nprint(f'   Model name: {model_name}')\nprint(f'   Epochs: {EPOCHS}')\nprint(f'   Batch size: {BATCH_SIZE}')\nprint(f'   Training samples: {len(X_train):,}')\nprint(f'   Validation samples: {len(X_val):,}')\nprint(f'   Data augmentation: Enabled')\nprint(f'\\n🚀 Ready to train!')

In [ ]:
# ============================================================================\n# CELL 6: TRAIN MODEL\n# ============================================================================\nprint('🚀 Starting training...')\nprint('=' * 70)\nprint('\\n⏰ This may take 30-60 minutes depending on your hardware...')\nprint('💡 Watch the progress bars and accuracy metrics below\\n')\n\n# Train model\nhistory = model.fit(\n    datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),\n    steps_per_epoch=len(X_train) // BATCH_SIZE,\n    epochs=EPOCHS,\n    validation_data=(X_val, y_val),\n    callbacks=callbacks,\n    verbose=1\n)\n\nprint('\\n' + '=' * 70)\nprint('✅ Training completed!')\nprint('=' * 70)

In [ ]:
# ============================================================================\n# CELL 7: VISUALIZE TRAINING HISTORY\n# ============================================================================\nprint('📊 Creating training visualizations...\\n')\n\nfig, axes = plt.subplots(2, 2, figsize=(15, 10))\n\n# Accuracy plot\naxes[0, 0].plot(history.history['accuracy'], label='Training', linewidth=2, marker='o', markersize=4)\naxes[0, 0].plot(history.history['val_accuracy'], label='Validation', linewidth=2, marker='s', markersize=4)\naxes[0, 0].set_title('Model Accuracy Over Time', fontsize=14, fontweight='bold')\naxes[0, 0].set_xlabel('Epoch', fontsize=12)\naxes[0, 0].set_ylabel('Accuracy', fontsize=12)\naxes[0, 0].legend(fontsize=11)\naxes[0, 0].grid(True, alpha=0.3)\n\n# Loss plot\naxes[0, 1].plot(history.history['loss'], label='Training', linewidth=2, marker='o', markersize=4)\naxes[0, 1].plot(history.history['val_loss'], label='Validation', linewidth=2, marker='s', markersize=4)\naxes[0, 1].set_title('Model Loss Over Time', fontsize=14, fontweight='bold')\naxes[0, 1].set_xlabel('Epoch', fontsize=12)\naxes[0, 1].set_ylabel('Loss', fontsize=12)\naxes[0, 1].legend(fontsize=11)\naxes[0, 1].grid(True, alpha=0.3)\n\n# Accuracy comparison\nfinal_train_acc = history.history['accuracy'][-1] * 100\nfinal_val_acc = history.history['val_accuracy'][-1] * 100\naxes[1, 0].bar(['Training', 'Validation'], [final_train_acc, final_val_acc], \n               color=['#2ecc71', '#3498db'], width=0.5)\naxes[1, 0].set_title('Final Accuracy Comparison', fontsize=14, fontweight='bold')\naxes[1, 0].set_ylabel('Accuracy (%)', fontsize=12)\naxes[1, 0].set_ylim(0, 100)\naxes[1, 0].grid(axis='y', alpha=0.3)\nfor i, v in enumerate([final_train_acc, final_val_acc]):\n    axes[1, 0].text(i, v + 2, f'{v:.2f}%', ha='center', fontsize=12, fontweight='bold')\n\n# Training progress\nepochs_range = range(1, len(history.history['accuracy']) + 1)\naxes[1, 1].plot(epochs_range, history.history['accuracy'], label='Train Acc', linewidth=2)\naxes[1, 1].plot(epochs_range, history.history['val_accuracy'], label='Val Acc', linewidth=2)\naxes[1, 1].fill_between(epochs_range, history.history['accuracy'], \n                        history.history['val_accuracy'], alpha=0.2)\naxes[1, 1].set_title('Training vs Validation Gap', fontsize=14, fontweight='bold')\naxes[1, 1].set_xlabel('Epoch', fontsize=12)\naxes[1, 1].set_ylabel('Accuracy', fontsize=12)\naxes[1, 1].legend(fontsize=11)\naxes[1, 1].grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.savefig(f'{model_name}_training_history.png', dpi=300, bbox_inches='tight')\nplt.show()\n\nprint(f'✅ Training history saved: {model_name}_training_history.png')\nprint(f'\\n📈 Final Results:')\nprint(f'   Training Accuracy: {final_train_acc:.2f}%')\nprint(f'   Validation Accuracy: {final_val_acc:.2f}%')\nprint(f'   Total Epochs: {len(history.history["accuracy"])}')

In [ ]:
# ============================================================================\n# CELL 8: EVALUATE MODEL ON TEST SET\n# ============================================================================\nprint('🧪 Evaluating model on test set...\\n')\n\n# Evaluate\ntest_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)\n\n# Predictions\ny_pred = model.predict(X_test, verbose=0)\ny_pred_classes = np.argmax(y_pred, axis=1)\ny_true_classes = np.argmax(y_test, axis=1)\n\nprint(f'📊 TEST RESULTS:')\nprint(f'   Test Loss: {test_loss:.4f}')\nprint(f'   Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')\n\n# Per-emotion accuracy\nprint(f'\\n📈 Per-Emotion Accuracy:')\nemotion_accuracies = []\nfor i, emotion in enumerate(EMOTIONS):\n    mask = y_true_classes == i\n    if mask.sum() > 0:\n        emotion_acc = accuracy_score(y_true_classes[mask], y_pred_classes[mask])\n        emotion_accuracies.append(emotion_acc * 100)\n        print(f'   {emotion:10s}: {emotion_acc*100:5.2f}%')\n    else:\n        emotion_accuracies.append(0)\n\n# Classification report\nprint(f'\\n📋 Detailed Classification Report:')\nreport = classification_report(y_true_classes, y_pred_classes, \n                               target_names=EMOTIONS, digits=4)\nprint(report)\n\n# Confusion matrix\ncm = confusion_matrix(y_true_classes, y_pred_classes)\n\nprint('\\n✅ Evaluation complete!')

In [ ]:
# ============================================================================\n# CELL 9: CONFUSION MATRIX VISUALIZATION\n# ============================================================================\nprint('📊 Creating confusion matrix visualization...\\n')\n\nfig, axes = plt.subplots(1, 2, figsize=(16, 6))\n\n# Confusion matrix (counts)\nsns.heatmap(cm, annot=True, fmt='d', cmap='Blues', \n            xticklabels=EMOTIONS, yticklabels=EMOTIONS,\n            cbar_kws={'label': 'Count'}, ax=axes[0])\naxes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')\naxes[0].set_xlabel('Predicted Emotion', fontsize=12)\naxes[0].set_ylabel('True Emotion', fontsize=12)\n\n# Confusion matrix (normalized)\ncm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]\nsns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='RdYlGn', \n            xticklabels=EMOTIONS, yticklabels=EMOTIONS,\n            cbar_kws={'label': 'Proportion'}, ax=axes[1], vmin=0, vmax=1)\naxes[1].set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')\naxes[1].set_xlabel('Predicted Emotion', fontsize=12)\naxes[1].set_ylabel('True Emotion', fontsize=12)\n\nplt.tight_layout()\nplt.savefig(f'{model_name}_confusion_matrix.png', dpi=300, bbox_inches='tight')\nplt.show()\n\nprint(f'✅ Confusion matrix saved: {model_name}_confusion_matrix.png')\n\n# Identify most confused pairs\nprint(f'\\n🔍 Most Confused Emotion Pairs:')\nconfusion_pairs = []\nfor i in range(len(EMOTIONS)):\n    for j in range(len(EMOTIONS)):\n        if i != j and cm[i, j] > 0:\n            confusion_pairs.append((EMOTIONS[i], EMOTIONS[j], cm[i, j]))\n\nconfusion_pairs.sort(key=lambda x: x[2], reverse=True)\nfor true_emotion, pred_emotion, count in confusion_pairs[:5]:\n    print(f'   {true_emotion:10s} → {pred_emotion:10s}: {count:4d} times')

In [ ]:
# ============================================================================\n# CELL 10: PER-EMOTION STATISTICS\n# ============================================================================\nprint('📊 Creating per-emotion statistics...\\n')\n\nfig, axes = plt.subplots(2, 2, figsize=(15, 10))\n\n# Per-emotion accuracy bar chart\ncolors = ['#e74c3c' if acc < 50 else '#f39c12' if acc < 70 else '#2ecc71' \n          for acc in emotion_accuracies]\nbars = axes[0, 0].bar(EMOTIONS, emotion_accuracies, color=colors, edgecolor='black', linewidth=1.5)\naxes[0, 0].set_title('Accuracy by Emotion', fontsize=14, fontweight='bold')\naxes[0, 0].set_xlabel('Emotion', fontsize=12)\naxes[0, 0].set_ylabel('Accuracy (%)', fontsize=12)\naxes[0, 0].set_ylim(0, 100)\naxes[0, 0].grid(axis='y', alpha=0.3)\naxes[0, 0].tick_params(axis='x', rotation=45)\n\n# Add value labels\nfor bar, acc in zip(bars, emotion_accuracies):\n    height = bar.get_height()\n    axes[0, 0].text(bar.get_x() + bar.get_width()/2., height + 1,\n                    f'{acc:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')\n\n# Sample distribution\ntest_emotion_counts = [np.sum(y_true_classes == i) for i in range(len(EMOTIONS))]\naxes[0, 1].bar(EMOTIONS, test_emotion_counts, color='skyblue', edgecolor='black', linewidth=1.5)\naxes[0, 1].set_title('Test Set Distribution', fontsize=14, fontweight='bold')\naxes[0, 1].set_xlabel('Emotion', fontsize=12)\naxes[0, 1].set_ylabel('Sample Count', fontsize=12)\naxes[0, 1].grid(axis='y', alpha=0.3)\naxes[0, 1].tick_params(axis='x', rotation=45)\n\n# Precision, Recall, F1-Score\nfrom sklearn.metrics import precision_recall_fscore_support\nprecision, recall, f1, _ = precision_recall_fscore_support(y_true_classes, y_pred_classes, \n                                                            average=None, labels=range(len(EMOTIONS)))\n\nx = np.arange(len(EMOTIONS))\nwidth = 0.25\naxes[1, 0].bar(x - width, precision * 100, width, label='Precision', color='#3498db')\naxes[1, 0].bar(x, recall * 100, width, label='Recall', color='#e74c3c')\naxes[1, 0].bar(x + width, f1 * 100, width, label='F1-Score', color='#2ecc71')\naxes[1, 0].set_title('Precision, Recall, F1-Score by Emotion', fontsize=14, fontweight='bold')\naxes[1, 0].set_xlabel('Emotion', fontsize=12)\naxes[1, 0].set_ylabel('Score (%)', fontsize=12)\naxes[1, 0].set_xticks(x)\naxes[1, 0].set_xticklabels(EMOTIONS, rotation=45)\naxes[1, 0].legend(fontsize=10)\naxes[1, 0].grid(axis='y', alpha=0.3)\naxes[1, 0].set_ylim(0, 100)\n\n# Overall metrics\noverall_metrics = {\n    'Accuracy': test_accuracy * 100,\n    'Avg Precision': np.mean(precision) * 100,\n    'Avg Recall': np.mean(recall) * 100,\n    'Avg F1-Score': np.mean(f1) * 100\n}\naxes[1, 1].bar(overall_metrics.keys(), overall_metrics.values(), \n               color=['#9b59b6', '#3498db', '#e74c3c', '#2ecc71'], \n               edgecolor='black', linewidth=1.5)\naxes[1, 1].set_title('Overall Model Performance', fontsize=14, fontweight='bold')\naxes[1, 1].set_ylabel('Score (%)', fontsize=12)\naxes[1, 1].set_ylim(0, 100)\naxes[1, 1].grid(axis='y', alpha=0.3)\naxes[1, 1].tick_params(axis='x', rotation=45)\n\nfor i, (metric, value) in enumerate(overall_metrics.items()):\n    axes[1, 1].text(i, value + 2, f'{value:.2f}%', ha='center', fontsize=10, fontweight='bold')\n\nplt.tight_layout()\nplt.savefig(f'{model_name}_statistics.png', dpi=300, bbox_inches='tight')\nplt.show()\n\nprint(f'✅ Statistics saved: {model_name}_statistics.png')

In [ ]:
# ============================================================================\n# CELL 11: SAVE TRAINED MODEL\n# ============================================================================\nprint('💾 Saving trained model...\\n')\n\n# Save final model\nfinal_model_path = f'{model_name}_final.h5'\nmodel.save(final_model_path)\nprint(f'✅ Model saved: {final_model_path}')\n\n# Save to server directory\nos.makedirs('server', exist_ok=True)\nserver_model_path = 'server/emotion_model_trained.h5'\nmodel.save(server_model_path)\nprint(f'✅ Server model saved: {server_model_path}')\n\n# Save metadata\nimport json\nmetadata = {\n    'model_name': model_name,\n    'dataset': 'FER2013-Enhanced',\n    'emotions': EMOTIONS,\n    'num_classes': len(EMOTIONS),\n    'img_size': IMG_SIZE,\n    'test_accuracy': float(test_accuracy),\n    'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),\n    'framework': 'TensorFlow/Keras',\n    'total_parameters': int(model.count_params()),\n    'epochs_trained': len(history.history['accuracy']),\n    'final_train_accuracy': float(history.history['accuracy'][-1]),\n    'final_val_accuracy': float(history.history['val_accuracy'][-1]),\n    'per_emotion_accuracy': {emotion: float(acc) for emotion, acc in zip(EMOTIONS, emotion_accuracies)}\n}\n\nmetadata_path = f'{model_name}_metadata.json'\nwith open(metadata_path, 'w') as f:\n    json.dump(metadata, f, indent=2)\nprint(f'✅ Metadata saved: {metadata_path}')\n\n# Save training history\nimport pickle\nhistory_path = f'{model_name}_history.pkl'\nwith open(history_path, 'wb') as f:\n    pickle.dump(history.history, f)\nprint(f'✅ Training history saved: {history_path}')\n\nprint(f'\\n📦 All files saved successfully!')\nprint(f'\\n📁 Generated Files:')\nprint(f'   1. {final_model_path}')\nprint(f'   2. {server_model_path}')\nprint(f'   3. {metadata_path}')\nprint(f'   4. {history_path}')\nprint(f'   5. {model_name}_training_history.png')\nprint(f'   6. {model_name}_confusion_matrix.png')\nprint(f'   7. {model_name}_statistics.png')

In [ ]:
# ============================================================================\n# CELL 12: TEST WITH SAMPLE IMAGES\n# ============================================================================\nprint('📸 Testing with sample images from test set...\\n')\n\n# Select random samples from each emotion\nfig, axes = plt.subplots(2, 7, figsize=(16, 5))\n\nfor i, emotion in enumerate(EMOTIONS):\n    # Get samples of this emotion\n    emotion_indices = np.where(y_true_classes == i)[0]\n    \n    if len(emotion_indices) > 0:\n        # Pick random sample\n        idx = np.random.choice(emotion_indices)\n        \n        # Get prediction\n        pred_probs = y_pred[idx]\n        pred_class = y_pred_classes[idx]\n        pred_emotion = EMOTIONS[pred_class]\n        confidence = pred_probs[pred_class] * 100\n        \n        # Display image\n        img = X_test[idx].reshape(IMG_SIZE, IMG_SIZE)\n        axes[0, i].imshow(img, cmap='gray')\n        axes[0, i].set_title(f'True: {emotion}', fontsize=10, fontweight='bold')\n        axes[0, i].axis('off')\n        \n        # Display prediction\n        color = 'green' if pred_emotion == emotion else 'red'\n        axes[1, i].bar(range(len(EMOTIONS)), pred_probs * 100, color='lightblue')\n        axes[1, i].bar(pred_class, pred_probs[pred_class] * 100, color=color)\n        axes[1, i].set_title(f'Pred: {pred_emotion}\\n{confidence:.1f}%', \n                            fontsize=9, fontweight='bold', color=color)\n        axes[1, i].set_xticks(range(len(EMOTIONS)))\n        axes[1, i].set_xticklabels([e[0].upper() for e in EMOTIONS], fontsize=8)\n        axes[1, i].set_ylim(0, 100)\n        axes[1, i].tick_params(axis='y', labelsize=7)\n\nplt.suptitle('Sample Predictions from Test Set', fontsize=16, fontweight='bold', y=1.02)\nplt.tight_layout()\nplt.savefig(f'{model_name}_sample_predictions.png', dpi=300, bbox_inches='tight')\nplt.show()\n\nprint(f'✅ Sample predictions saved: {model_name}_sample_predictions.png')\n\n# Show some correct and incorrect predictions\ncorrect_mask = y_true_classes == y_pred_classes\ncorrect_count = np.sum(correct_mask)\nincorrect_count = len(y_true_classes) - correct_count\n\nprint(f'\\n📊 Prediction Summary:')\nprint(f'   Correct predictions: {correct_count:,} ({correct_count/len(y_true_classes)*100:.2f}%)')\nprint(f'   Incorrect predictions: {incorrect_count:,} ({incorrect_count/len(y_true_classes)*100:.2f}%)')

## 🎉 Training Complete!\n\n### Summary:\n\n✅ **Model trained successfully** on FER2013 Enhanced dataset\n✅ **Visualizations created**: Training curves, confusion matrix, statistics\n✅ **Model saved** to multiple locations\n✅ **Metadata exported** for future reference\n\n### Key Metrics:\n- **Dataset**: 35,887 images, 7 emotions\n- **Test Accuracy**: Check Cell 8 output\n- **Per-Emotion Performance**: See Cell 10 visualizations\n\n### Generated Files:\n1. `*_final.h5` - Trained model\n2. `server/emotion_model_trained.h5` - Server-ready model\n3. `*_metadata.json` - Model information\n4. `*_training_history.png` - Training curves\n5. `*_confusion_matrix.png` - Confusion matrix\n6. `*_statistics.png` - Per-emotion statistics\n7. `*_sample_predictions.png` - Sample test predictions\n\n### Next Steps:\n1. ✅ Review the visualizations above\n2. ✅ Check per-emotion accuracy in Cell 10\n3. ✅ Use the saved model in your application\n4. ✅ Test with real images using `emotion_detection_notebook.ipynb`\n\n### To Improve Accuracy Further:\n- Train for more epochs (increase EPOCHS in Cell 5)\n- Use advanced techniques from `train_high_accuracy_fer2013.py`\n- Add more data augmentation\n- Try transfer learning with pre-trained models\n\n### Documentation:\n- Training guide: `HIGH_ACCURACY_TRAINING_GUIDE_NEPALI.md`\n- Usage guide: `JUPYTER_NOTEBOOK_GUIDE_NEPALI.md`\n- Model info: `AURABOT_MODEL_TRAINING_SUMMARY_NEPALI.md`